# Model Training: Neural Machine Translation (Ensemble)

## Overview

This notebook trains multiple T5 and mT5 models for ensemble inference.

Using the augmented dataset (~28,000 training examples), we train:
- **3 T5-small models** with different random seeds
- **1 mT5-small model** for diversity in the ensemble

This creates a 4-model ensemble for more robust predictions.

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import T5Tokenizer, T5ForConditionalGeneration, MT5ForConditionalGeneration, AutoTokenizer, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import os
import numpy as np
import random

os.environ["TOKENIZERS_PARALLELISM"] = "false"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print("\nLoading augmented data...")
train_df = pd.read_csv('train_augmented.csv')
test_df = pd.read_csv('test.csv')

print(f"Augmented train size: {len(train_df)}")
print(f"Test size: {len(test_df)}")

print("\n--- Data Sources ---")
print(train_df['source'].value_counts())

In [ ]:
train_df = train_df.dropna(subset=['transliteration', 'translation'])
print(f"After dropping NA: {len(train_df)}")

## 1. Create Dataset and Tokenization

In [ ]:
train_texts = train_df['transliteration'].tolist()
train_labels = train_df['translation'].tolist()

train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_texts, train_labels, test_size=0.05, random_state=42
)

print(f"Training samples: {len(train_texts)}")
print(f"Validation samples: {len(val_texts)}")

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = "translate Akkadian to English: " + str(self.texts[idx])
        label = str(self.labels[idx])
        
        source = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        target = self.tokenizer(
            label,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': source['input_ids'].squeeze(),
            'attention_mask': source['attention_mask'].squeeze(),
            'labels': target['input_ids'].squeeze()
        }

print("Loading tokenizer...")
t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
mt5_tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

print("Creating datasets...")
train_dataset = TranslationDataset(train_texts, train_labels, t5_tokenizer, max_len=128)
val_dataset = TranslationDataset(val_texts, val_labels, t5_tokenizer, max_len=128)

## 2. Train Ensemble Models

In [ ]:
def train_model(model_name, model_class, base_model, tokenizer, output_dir, seed=42):
    """Train a single model and save it."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    
    print(f"\n{'='*50}")
    print(f"Training {model_name} (seed={seed})")
    print(f"{'='*50}")
    
    model = model_class.from_pretrained(base_model)
    model = model.to(DEVICE)
    
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        warmup_steps=100,
        weight_decay=0.01,
        logging_steps=100,
        eval_strategy="steps",
        eval_steps=500,
        save_strategy="steps",
        save_steps=500,
        load_best_model_at_end=True,
        report_to="none",
        learning_rate=3e-4,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        seed=seed,
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
    )
    
    print(f"Starting training for {model_name}...")
    trainer.train()
    
    os.makedirs("./ensemble_models", exist_ok=True)
    save_path = f"./ensemble_models/{model_name}"
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f"Saved {model_name} to {save_path}")
    
    return model

trained_models = {}

for i in range(3):
    model_name = f"t5_small_seed_{i+1}"
    model = train_model(
        model_name,
        T5ForConditionalGeneration,
        "t5-small",
        t5_tokenizer,
        f"./results_t5_seed_{i+1}",
        seed=42 + i * 10
    )
    trained_models[model_name] = model

print("\n" + "="*50)
print("Training mT5-small model...")
print("="*50)
mt5_model = train_model(
    "mt5_small",
    MT5ForConditionalGeneration,
    "google/mt5-small",
    mt5_tokenizer,
    "./results_mt5",
    seed=42
)
trained_models["mt5_small"] = mt5_model

print("\n" + "="*50)
print("All 4 models trained and saved to ./ensemble_models/")
print("="*50)

## 3. Summary

In [ ]:
print("="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Models trained: {len(trained_models)}")
print(f"Models: {list(trained_models.keys())}")
print(f"Device: {DEVICE}")
print(f"\nAll models saved to: ./ensemble_models/")
print("\nEnsemble models ready for inference in 08_load_model_submission.ipynb")